In [1]:
# Import relevant libraries.
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime
import re

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('words')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.corpus import words
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package words to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Jai
[nltk_data]     T\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Load dataset. Change directory as required.
df = pd.read_csv('us_speeches.csv')

In [3]:
df.head()

,reference,country,date,title,author,is_gov,text
0,r970105a_FOMC,united states,05/01/1997,I. Structural Models and Monetary Policy Analysis,meyer,0,I am in the middle of my third interesting and...
1,r970114a_FOMC,united states,14/01/1997,National Bank of Belgium,greenspan,1,"Mr. Prime Minister, Minister of Finance, Minis..."
2,r970116a_FOMC,united states,16/01/1997,Balanced Risks Going Forward,meyer,0,Measured on a fourth quarter to fourth quarter...
3,r970124a_FOMC,united states,24/01/1997,Activities,meyer,0,My topic this morning is financial modernizati...
4,r970128a_FOMC,united states,28/01/1997,should,phillips,0,Good afternoon. It is a pleasure to be here to...


In [4]:
df.country.value_counts()

united states    1550
Name: country, dtype: int64

In [5]:
# Add a column to calculate the string length per speech.
df['len'] = df['text'].str.len()
df

,reference,country,date,title,author,is_gov,text,len
0,r970105a_FOMC,united states,05/01/1997,I. Structural Models and Monetary Policy Analysis,meyer,0,I am in the middle of my third interesting and...,20327
1,r970114a_FOMC,united states,14/01/1997,National Bank of Belgium,greenspan,1,"Mr. Prime Minister, Minister of Finance, Minis...",31719
2,r970116a_FOMC,united states,16/01/1997,Balanced Risks Going Forward,meyer,0,Measured on a fourth quarter to fourth quarter...,25746
3,r970124a_FOMC,united states,24/01/1997,Activities,meyer,0,My topic this morning is financial modernizati...,30376
4,r970128a_FOMC,united states,28/01/1997,should,phillips,0,Good afternoon. It is a pleasure to be here to...,18196
...,...,...,...,...,...,...,...,...
1545,r221010a_FOMC,united states,10/10/2022,Restoring Price Stability in an Uncertain Econ...,brainard,0,It is a pleasure to join this discussion today...,11505
1546,r221012b_FOMC,united states,12/10/2022,Managing the Promise and Risk of Financial Inn...,barr,0,"Thank you, Chris, and thank you for the invita...",12816
1547,r221012a_FOMC,united states,12/10/2022,Forward Guidance as a Monetary Policy Tool: Co...,bowman,0,Thanks to the Money Marketeers for inviting me...,18600
1548,r221014a_FOMC,united states,14/10/2022,The U.S. Dollar and Central Bank Digital Curre...,waller,0,"Thank you, Professor Jackson, and thank you to...",14060


In [6]:
# Text cleaning (Convert to lower case and remove punctuation)
df['text'] = df['text'].str.lower().str.replace('[^\w\s]', '', regex=True)

In [7]:
# VADER sentiment (Calculate Sentiment intensity analysis using Vadar sentiment)
sia = SentimentIntensityAnalyzer()
df[['neg', 'neu', 'pos', 'compound']] = df['text'].apply(lambda x: pd.Series(sia.polarity_scores(x)))

In [8]:
# TextBlob sentiment (Calculate polarity and subjectivity using TextBlob)
df[['polarity','subjectivity']] = df['text'].apply(lambda x: pd.Series(TextBlob(x).sentiment))

In [9]:
# Load Loughran–McDonald Dictionary
lm_dict = pd.read_csv("LM_dictionary.csv")  
print("LM Columns:", lm_dict.columns)  # check columns

LM Columns: Index(['Word', 'Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong',
       'Weak', 'Constraining'],
      dtype='object')


In [10]:
# Create a mapping: Word -> list of categories
lm_dict_map = {}
for _, row in lm_dict.iterrows():
    word = row['Word'].upper()
    categories = [col for col in lm_dict.columns[1:] if row[col] > 0]  # skip 'Word' column
    lm_dict_map[word] = categories

In [11]:
# Function to compute LM sentiment
def lm_sentiment(text, lm_dict_map):
    words = re.findall(r'\b\w+\b', text.upper())
    pos = sum(1 for w in words if 'Positive' in lm_dict_map.get(w, []))
    neg = sum(1 for w in words if 'Negative' in lm_dict_map.get(w, []))
    total = pos + neg
    return 0 if total == 0 else (pos - neg)/total

In [12]:
#import re
# Apply LM sentiment
df['lm_score'] = df['text'].apply(lambda x: lm_sentiment(x, lm_dict_map))

In [13]:
# Combined Score (simple average)
df['combined_score'] = (df['compound'] + df['polarity'] + df['lm_score']) / 3

In [14]:
# LM Label Thresholds
def lm_label(score, pos_thresh=0.05, neg_thresh=-0.05):
    """
    Assigns a sentiment label based on LM score thresholds.
    
    Parameters:
        score: LM sentiment score ([-1,1])
        pos_thresh: threshold above which text is Positive
        neg_thresh: threshold below which text is Negative
        
    Returns:
        'Positive', 'Negative', or 'Neutral'
    """
    if score > pos_thresh:
        return "Positive"
    elif score < neg_thresh:
        return "Negative"
    else:
        return "Neutral"
    
df['lm_label'] = df['lm_score'].apply(lambda x: lm_label(x))    

# Export Selected Columns
columns_to_export = [
    'reference','date','text', 
    'neg', 'neu', 'pos', 'compound', 
    'polarity', 'subjectivity', 
    'lm_score', 
    'combined_score', 
    'lm_label'
]

In [15]:
# Export CSV
df.to_csv("us_sentiment_labeled.csv", columns=columns_to_export, index=False, encoding='utf-8')

In [16]:
# Export Excel
df.to_excel("us_sentiment_labeled.xlsx", columns=columns_to_export, index=False, sheet_name="US_Sentiment")